# Do do tren CA HAI nhanh (anh + van ban) + nhanh GOP

Chi FORWARD, khong train. Doi `JOB` o Cell 2 roi Run All. Moi lan chay DUNG mot muc quen.

| JOB | Bo | Muc quen | Checkpoint can co |
|---|---|---|---|
| `m3`  | MIMIC | 3%  | og · re · fmi_m3 · p3_m3 · abl_fila · abl_ihl · abl_mumr |
| `m6`  | MIMIC | 6%  | og · re · fmi_m6 · p3_m6 |
| `m10` | MIMIC | 10% | og · re · fmi_m10 · p3_m10 |
| `iu`  | IU    | 3%  | og · re · fmi_iu · p3_iu |

## Vi sao chay cai nay

Moi do do trong bang hien tai chi doc `outputs[1] = logits_img`, dung nhu ban cai dat goc
cua Forget-MI. Bai bao KHONG neu dung nhanh nao. Voi mot phuong phap go bo DA PHUONG THUC
thi do bang mot nhanh la do thieu — ke tan cong that co ca hai dau ra.

Moi checkpoint xuat **3 dong**:

- `img` — nhanh anh. **PHAI trung so da bao cao** → day la phep TU KIEM, lech la loi code.
- `txt` — nhanh van ban, chua tung duoc do.
- `fuse` — gop: `CE_img + CE_txt`, xac suat = trung binh hai softmax, **MIA = SVM tren
  vector 2 chieu `[CE_img, CE_txt]`**.

Bang chinh trong khoa luan **giu nguyen nhanh anh**. Bang moi la BO SUNG, dat canh.

## Thoi gian uoc tinh

~8 phut / checkpoint, rieng P3 va ablation cong them ~5 phut vi phai chay lai Fisher+FILA
de tai tao W* (checkpoint P3 chi chua LoRA). JOB `m3` co 7 checkpoint → ~1,2h.


In [ ]:
# Cell 1: setup
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/eval_multimodal.py'), \
    'Chua co training/eval_multimodal.py -> git push code moi roi Save Version lai!'
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())

# Tu kiem cac ham thuan-mang truoc khi ton GPU (30 giay)
print('\n--- tu kiem eval_multimodal ---')
subprocess.run(['python','tools/test_eval_multimodal.py'],check=True)


In [ ]:
# Cell 2: CHON JOB + tu do checkpoint
import glob, os, re
JOB  = 'm3'      # m3 | m6 | m10 | iu
SEED = 42

JOBS = {'m3': ('mimic', 3), 'm6': ('mimic', 6), 'm10': ('mimic', 10), 'iu': ('iu', 3)}
assert JOB in JOBS, f'JOB phai thuoc {sorted(JOBS)}'
DATASET, PCT = JOBS[JOB]

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)

if DATASET=='mimic':
    CONFIG='config_advanced_kaggle.yaml'
    DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
    assert DATA and MOD,'Add forget-mi-data + forget-mi-models-full'
    BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gh=[b for b in bins(MOD) if f'model_retrained_{PCT}per' in b]
    assert gh,f'Khong thay model_retrained_{PCT}per'
    GOLD=os.path.dirname(gh[0])
    TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
    SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET=f'./data_splits/forget_set_{PCT}per.csv'
else:
    CONFIG='config_loku_iu_kaggle.yaml'
    DATA=fd('forget-mi-data-iu'); MOD=fd('forget-mi-models-iu'); MODRE=fd('forget-mi-models-iu-re')
    RAD=fd('chest-xrays-indiana-university')
    assert DATA and MOD and RAD,'Add forget-mi-data-iu + forget-mi-models-iu(+ -re) + raddar'
    ogb=[b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    GOLD=os.path.dirname(reb[0]) if reb else BASE
    tsv=glob.glob(os.path.join(DATA,'**','all_data.tsv'),recursive=True) or glob.glob('/kaggle/input/**/all_data.tsv',recursive=True)
    TEXT=os.path.dirname(tsv[0])
    IMG=(glob.glob(os.path.join(RAD,'**','images_normalized'),recursive=True) or [RAD])[0]
    SPLIT=(glob.glob(os.path.join(DATA,'**','iu-split.csv'),recursive=True)+glob.glob('/kaggle/input/**/iu-split.csv',recursive=True))[0]
    FORGET=(glob.glob(os.path.join(DATA,'**',f'forget_set_{PCT}per_iu.csv'),recursive=True)+
            glob.glob(f'/kaggle/input/**/forget_set_{PCT}per_iu.csv',recursive=True))[0]

for n,p in {'BASE':BASE,'GOLD':GOLD,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'

RESULTS=f'/kaggle/working/results_multimodal_{JOB}.csv'
COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'output_dir':f'/kaggle/working/mm_{JOB}','use_noise':1}

# ---------- tu do checkpoint trong moi input da attach ----------
# FMI/NegGrad/CF-k/EU-k -> .pth day du   |   P3 + ablation -> .pt chi chua LoRA
# f3/f4 = hai ung vien tang luc quen, chi co o m3 (xem EXTRA_OVR ben duoi).
TAGS = {'m3':  {'fmi':'fmi_m3','p3':'p3_m3','abl_fila':'abl_fila','abl_ihl':'abl_ihl',
                'abl_mumr':'abl_mumr','f3':'p3_f3','f4':'p3_f4'},
        'm6':  {'fmi':'fmi_m6','p3':'p3_m6'},
        'm10': {'fmi':'fmi_m10','p3':'p3_m10'},
        'iu':  {'fmi':'fmi_iu','p3':'p3_iu'}}[JOB]

cands = [p for p in glob.glob('/kaggle/input/**/*.pt',recursive=True)
              + glob.glob('/kaggle/input/**/*.pth',recursive=True)
         if os.path.getsize(p) > 1e5]
print(f'Tim thay {len(cands)} file checkpoint duoi /kaggle/input:')
for p in sorted(cands): print(f'   {os.path.getsize(p)/1e6:8.1f} MB  {p}')

def pick(tag, want_lora):
    """Chon file khop tag; uu tien last/latest (E30) roi den selected."""
    hits=[p for p in cands if tag in p.replace('\\','/')]
    hits=[p for p in hits if (p.endswith('.pt') if want_lora else p.endswith('.pth'))]
    if not hits: return None
    def rank(p):
        b=os.path.basename(p).lower()
        return (0 if ('last' in b or 'latest' in b or 'epoch_29' in p) else
                1 if 'selected' in b else 2, len(p))
    return sorted(hits,key=rank)[0]

# (label, model_type, path). og/re luon co san tu dataset model.
PLAN=[('og','pretrained',BASE), ('re','pretrained',GOLD)]
for lab,tag in TAGS.items():
    lora = lab!='fmi'
    p = pick(tag, want_lora=lora)
    PLAN.append((lab,'p3_lora' if lora else 'state_dict',p))

# ---------- override RIENG tung checkpoint ----------
# BAT BUOC voi abl_fila: no duoc train voi loku_random_init=True (khong co FILA), nen khi
# dung lai PHAI bao lai co nay. Neu quen, adv_common se tru FILA vao nen -> W* KHAC luc
# train -> moi so deu sai ma khong bao loi. Ba ablation con lai chi doi TRONG SO LOSS,
# khong anh huong gi den viec dung lai model nen khong can override.
#
# f3/f4 CUNG BAT BUOC — va nguy hiem hon abl_fila vi HONG AM THAM:
#   infer_lora_cfg chi suy duoc lora_extra_target_modules + lora_image_last_k_blocks tu
#   TEN KHOA cua checkpoint. Hai he so tru FILA thi KHONG suy duoc tu dau, ma chung quyet
#   dinh W* = W - gamma*B*A*. Dung sai gamma -> nen khac luc train -> moi so sai, khong
#   assert nao bat duoc (assert chi so sanh TEN khoa va SO tensor).
#   Rieng lora_image_include_fc1 thi hong CO TIENG: checkpoint f4 co khoa img_model.fc1.*
#   ma model dung lai khong co cho -> eval_multimodal raise SystemExit "khoa khong co cho".
#   lora_image_last_k_blocks=3 ghi ra cho ro rang; infer se tu dat lai dung gia tri nay.
EXTRA_OVR = {
 'abl_fila': {'loku_random_init': 1},
 'f3': {'loku_subtract_scale': 1.5, 'loku_image_subtract_scale': 1.0},
 'f4': {'loku_subtract_scale': 1.5, 'loku_image_subtract_scale': 1.0,
        'lora_image_last_k_blocks': 3, 'lora_image_include_fc1': 1},
}

print(f'\n=== KE HOACH cho JOB={JOB} ({DATASET} {PCT}%) ===')
for lab,mt,p in PLAN:
    ex = EXTRA_OVR.get(lab)
    print(f'  {lab:9} {mt:11} {p if p else "*** KHONG TIM THAY — se BO QUA ***"}'
          + (f'   [+{ex}]' if ex else ''))
print('\nThieu cai nao thi attach them Output cua notebook do (Add Input -> Notebook Output),')
print('hoac sua tay bien PLAN o cell nay.')


In [ ]:
# Cell 3: CHAY — moi checkpoint 3 dong (img / txt / fuse)
import os, subprocess, time
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled',
     'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}

def run_one(label, mtype, path):
    ovr=dict(COMMON); ovr['id']=f'{label}_{JOB}'
    ovr.update(EXTRA_OVR.get(label, {}))     # vd abl_fila -> loku_random_init=1
    cmd=['python','training/eval_multimodal.py','--config',CONFIG,'--seed',str(SEED),
         '--label',label,'--model_type',mtype,'--model_path',path,
         '--out_csv',RESULTS,'--override',','.join(f'{k}={v}' for k,v in ovr.items())]
    print('='*72+f'\n{label}  ({mtype})\n'+'='*72)
    t0=time.time()
    try:
        subprocess.run(cmd,env=env,check=True)
        print(f'OK {label}  {(time.time()-t0)/60:.1f} phut')
    except subprocess.CalledProcessError as e:
        print(f'FAIL {label} rc={e.returncode} — chay tiep cai sau')

for lab,mt,p in PLAN:
    if p: run_one(lab,mt,p)
    else: print(f'(bo qua {lab} — khong co checkpoint)')


In [ ]:
# Cell 4: bang ket qua + TU KIEM
import os, pandas as pd
pd.set_option('display.width',250)
if not os.path.exists(RESULTS):
    print('chua co',RESULTS)
else:
    d=pd.read_csv(RESULTS)
    cols=['label','view','Df_AUC','Df_F1','Dt_AUC','Dt_F1','MIA','MIA_paper',
          'member_ce','nonmember_ce','forget_ce']
    for v in ['img','txt','fuse']:
        print(f'\n===== view = {v} =====')
        print(d[d['view']==v][cols].to_string(index=False))

    print('\n\n===== TU KIEM: cot view=img phai TRUNG so da bao cao =====')
    # (Df-AUC, MIA) tai E30. f3/f4 lay tu results_p3_f3/f4 (12/8).
    REF={'m3':  {'og':(0.731,0.657),'re':(0.498,0.423),'fmi':(0.623,0.627),'p3':(0.683,0.552),
                 'f3':(0.590,0.418),'f4':(0.528,0.368)},
         'm6':  {'og':(0.730,0.736),'re':(0.596,0.613),'fmi':(0.633,0.773),'p3':(0.670,0.706)},
         'm10': {'og':(0.757,0.717),'re':(0.547,0.364),'fmi':(0.704,0.616),'p3':(0.728,0.731)},
         'iu':  {'og':(1.000,0.990),'re':(0.651,0.545),'fmi':(0.854,0.581),'p3':(0.710,0.560)}}[JOB]
    img=d[d['view']=='img'].set_index('label')
    print(f"{'label':8} | {'Df-AUC moi':>10} {'cu':>7} | {'MIA moi':>8} {'cu':>7} | ket luan")
    print('-'*74)
    for lab,(ref_auc,ref_mia) in REF.items():
        if lab not in img.index: continue
        r=img.loc[lab]
        ok=abs(r['Df_AUC']-ref_auc)<0.005 and abs(r['MIA']-ref_mia)<0.005
        print(f"{lab:8} | {r['Df_AUC']:10.3f} {ref_auc:7.3f} | {r['MIA']:8.3f} {ref_mia:7.3f} | "
              f"{'TRUNG' if ok else '*** LECH -> bao lai, dung dung so txt/fuse ***'}")

    # ---- so sanh f3 vs f4 tren 3 khung nhin (viec can de CHOT cau hinh) ----
    if {'f3','f4'} <= set(d['label']):
        print('\n\n===== f3 vs f4 · ba khung nhin =====')
        piv=d[d['label'].isin(['re','f3','f4'])].pivot_table(
            index='view', columns='label',
            values=['Df_AUC','Dt_AUC','MIA','forget_ce'])
        print(piv.to_string())
        print('\nDoc: so cua f3/f4 cang gan cot "re" (gold) cang tot, o CA BA khung nhin.')

print('\nTAI VE: results_multimodal_<JOB>.csv')
